In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv('Combined Data.csv')
df = df.dropna()

#preproces secara sederhananya

X = df['statement']
y = df['status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#pipelinenya
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

joblib.dump(model, 'mental_health_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print(f"Model trained. Accuracy: {accuracy_score(y_test, model.predict(X_test_tfidf)):.2f}")

Model trained. Accuracy: 0.77


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
import time
import numpy as np

def measure_latency(text):
    start_time = time.time()


    text_tfidf = tfidf.transform([text])
    prediction = model.predict(text_tfidf)

    end_time = time.time()
    latency = end_time - start_time
    return latency, prediction


sample_text = "I feel very anxious and stressed today"
latency, res = measure_latency(sample_text)

print(f"Hasil Prediksi: {res[0]}")
print(f"Latency: {latency:.5f} detik")

Hasil Prediksi: Stress
Latency: 0.00246 detik


In [7]:
%%writefile test_script.py
import pytest
import joblib


model = joblib.load('mental_health_model.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

def predict_status(text):
    text_tfidf = tfidf.transform([text])
    return model.predict(text_tfidf)[0]


def test_prediction_not_empty():
    result = predict_status("I am happy")
    assert result is not None


def test_prediction_type():
    result = predict_status("I feel sad")
    assert isinstance(result, (str, int))


def test_consistency():
    input_text = "Panic attack"
    assert predict_status(input_text) == predict_status(input_text)

Writing test_script.py


In [8]:
!pytest test_script.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, langsmith-0.7.34, typeguard-4.5.1
collected 3 items                                                              

test_script.py ...                                                       [100%]

============================== 3 passed in 4.12s ===============================
